# ISTVT — Model Architecture
**Paper:** ISTVT: Interpretable Spatial-Temporal Video Transformer for Deepfake Detection  
Zhao et al., IEEE TIFS 2023

This notebook implements the **three novel contributions** of the paper:
1. Xception Entry-Flow Feature Extractor (Sec. III-A)
2. Decomposed Spatial-Temporal Self-Attention — Eq. 1 & 2 (Sec. III-B)
3. Self-Subtract Mechanism — Eq. 3 (Sec. III-C)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Part 1 — Feature Extractor: Xception Entry Flow
*(Paper Sec. III-A)*  
> *"we use a tiny convolutional network composed of several Xception blocks  
> (i.e. the entry flow of Xception network) to extract textural features"*

Applied to every frame independently. Output shape: `(B*T, embed_dim, fh, fw)`


In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        return self.pw(self.dw(x))


class XceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2):
        super().__init__()
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.main = nn.Sequential(
            nn.ReLU(),
            SeparableConv2d(in_ch, out_ch),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            SeparableConv2d(out_ch, out_ch),
            nn.BatchNorm2d(out_ch),
            nn.MaxPool2d(3, stride, 1),
        )

    def forward(self, x):
        return self.main(x) + self.skip(x)


class XceptionEntryFlow(nn.Module):
    """
    Entry flow of Xception — per-frame texture feature extractor.
    Input:  (B, 3, H, W)
    Output: (B, out_ch, H/8, W/8)
    """
    def __init__(self, base_ch=64, out_ch=256):
        super().__init__()
        self.out_ch = out_ch
        self.stem = nn.Sequential(
            nn.Conv2d(3, base_ch // 2, 3, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch // 2), nn.ReLU(),
            nn.Conv2d(base_ch // 2, base_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(base_ch), nn.ReLU(),
        )
        self.block1 = XceptionBlock(base_ch, base_ch * 2)
        self.block2 = XceptionBlock(base_ch * 2, base_ch * 4)
        self.block3 = XceptionBlock(base_ch * 4, out_ch)

    def forward(self, x):
        return self.block3(self.block2(self.block1(self.stem(x))))

print("XceptionEntryFlow defined ✓")

XceptionEntryFlow defined ✓


## Part 2 — Self-Subtract Mechanism
*(Paper Sec. III-C, Eq. 3, Fig. 4)*

**Eq. 3:**  `I' = cat( I[0:2], I[2:] − I[1:-1], dim=0 )`

- `I[0]` = temporal CLS token → **kept as-is**
- `I[1]` = first frame tokens → **kept as-is**
- `I[2:]` = remaining frames → **replaced by (frame_i − frame_{i−1})**

This forces temporal attention to focus on **change between frames**,  
not static content — highlighting inter-frame inconsistency.  
Applied only to **Q and K** of temporal attention; **V keeps original tokens**.


In [ ]:
def self_subtract(x):
    """
    Implements Eq. 3 from the paper.
    x: (B, T+1, HW+1, C)
    Returns x' of same shape with frame diffs replacing frames i >= 2.
    """
    keep  = x[:, 0:2]               # CLS + first frame — unchanged
    diffs = x[:, 2:] - x[:, 1:-1]  # frame_i - frame_{i-1}
    return torch.cat([keep, diffs], dim=1)

print("self_subtract defined ✓")

self_subtract defined ✓


## Part 3 — Decomposed Spatial-Temporal Self-Attention
*(Paper Sec. III-B, Eq. 1 & 2, Fig. 3)*

Instead of full attention over all T×HW tokens (cost: O(T²H²W²)), the paper decomposes into:

| Branch | Attends over | Fixed axis | Equation |
|---|---|---|---|
| **Temporal SA** | T frames | spatial position j | Eq. 1 |
| **Spatial SA** | HW patches | frame k | Eq. 2 |

Combined cost: **O(T² + H²W²)** — much cheaper.

Applied **temporal first** (best variant per Table III ablation in paper).


In [ ]:
class DecomposedSTAttention(nn.Module):
    """
    Core novel module of ISTVT.
    Temporal SA (with self-subtract on Q,K) -> Spatial SA, applied sequentially.
    """
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        # Temporal: Q,K from subtracted tokens; V from original tokens
        self.t_qk  = nn.Linear(dim, dim * 2, bias=False)
        self.t_v   = nn.Linear(dim, dim,     bias=False)
        self.t_out = nn.Linear(dim, dim,     bias=False)

        # Spatial: standard QKV
        self.s_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.s_out = nn.Linear(dim, dim,     bias=False)

    def _attn(self, q, k, v):
        """
        Multi-head attention over last axis of (B, G, L, C).
        G = grouped axis (not attended over), L = sequence axis.
        """
        B, G, L, C = q.shape
        H, D = self.num_heads, self.head_dim

        def to_heads(t):
            return t.view(B, G, L, H, D).permute(0, 1, 3, 2, 4)

        qh, kh, vh = to_heads(q), to_heads(k), to_heads(v)
        attn = (qh @ kh.transpose(-2, -1)) * self.scale    # (B,G,H,L,L)
        attn = attn.softmax(dim=-1)
        out  = (attn @ vh).permute(0, 1, 3, 2, 4).reshape(B, G, L, C)
        return out, attn

    def forward(self, x, return_attn=False):
        """
        x: (B, T+1, HW+1, C)
        """
        # ── Eq. 1: Temporal Self-Attention ──────────────────────────────────
        x_sub = self_subtract(x)            # apply self-subtract (Eq. 3)
        qk    = self.t_qk(x_sub)
        q_t, k_t = qk.chunk(2, dim=-1)
        v_t   = self.t_v(x)                # V from ORIGINAL tokens

        # rearrange: (B, T+1, HW+1, C) -> (B, HW+1, T+1, C)
        # so attention runs over T axis (Eq. 1)
        q_t = q_t.permute(0, 2, 1, 3)
        k_t = k_t.permute(0, 2, 1, 3)
        v_t = v_t.permute(0, 2, 1, 3)
        t_out, t_attn = self._attn(q_t, k_t, v_t)
        t_out = t_out.permute(0, 2, 1, 3)  # back to (B, T+1, HW+1, C)
        x = x + self.t_out(t_out)

        # ── Eq. 2: Spatial Self-Attention ───────────────────────────────────
        # x is (B, T+1, HW+1, C); attention runs over HW+1 axis natively
        qkv = self.s_qkv(x)
        q_s, k_s, v_s = qkv.chunk(3, dim=-1)
        s_out, s_attn = self._attn(q_s, k_s, v_s)
        x = x + self.s_out(s_out)

        if return_attn:
            return x, t_attn, s_attn
        return x

print("DecomposedSTAttention defined ✓")

DecomposedSTAttention defined ✓


## Part 4 — Transformer Block
Standard wrapper: LayerNorm → Decomposed ST-Attention → residual → FFN


In [ ]:
class STBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = DecomposedSTAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x, return_attn=False):
        if return_attn:
            attn_out, t_attn, s_attn = self.attn(self.norm1(x), return_attn=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        if return_attn:
            return x, t_attn, s_attn
        return x

print("STBlock defined ✓")

STBlock defined ✓


## Part 5 — Full ISTVT Model
*(Paper Sec. III-A, Fig. 2)*

Full pipeline:
```
Frame sequence (B,T,3,H,W)
  → Xception extractor (per frame)
  → Patch tokens + spatial CLS + temporal CLS + positional embedding
  → M × STBlock (decomposed ST-attention + self-subtract inside each)
  → MLP head on O(0,0,:)  ← joint CLS token
  → real/fake logit
```


In [ ]:
class ISTVT(nn.Module):
    def __init__(
        self,
        img_size=128,    # paper: 300  (scaled for single GPU)
        seq_len=6,       # paper: 6    (kept — short-term inconsistency)
        embed_dim=256,   # paper: 728  (scaled)
        depth=6,         # paper: 12   (scaled)
        num_heads=8,     # paper: 8    (kept)
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.seq_len = seq_len

        self.extractor = XceptionEntryFlow(base_ch=64, out_ch=embed_dim)
        self._embed_dim = embed_dim

        # compute spatial grid size produced by extractor
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            feat  = self.extractor(dummy)
            _, _, fh, fw = feat.shape
        self.fh, self.fw = fh, fw
        self.num_patches = fh * fw

        # CLS tokens (paper Sec. III-A)
        self.spatial_cls  = nn.Parameter(torch.zeros(1, seq_len, 1, embed_dim))
        self.temporal_cls = nn.Parameter(torch.zeros(1, 1, self.num_patches + 1, embed_dim))
        self.pos_embed    = nn.Parameter(torch.zeros(1, seq_len + 1, self.num_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.spatial_cls,  std=0.02)
        nn.init.trunc_normal_(self.temporal_cls, std=0.02)
        nn.init.trunc_normal_(self.pos_embed,    std=0.02)

        # M transformer blocks
        self.blocks = nn.ModuleList([
            STBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Prediction head (binary: real=0, fake=1)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1),
        )

    @property
    def embed_dim(self):
        return self._embed_dim

    def forward(self, x, return_attn=False):
        """
        x: (B, T, 3, H, W)
        returns logits (B, 1) — use BCEWithLogitsLoss for training
        """
        B, T, C, H, W = x.shape

        # 1. Per-frame feature extraction
        feats  = self.extractor(x.view(B * T, C, H, W))          # (B*T, D, fh, fw)
        feats  = feats.view(B, T, self.embed_dim, self.fh, self.fw)
        tokens = feats.flatten(3).permute(0, 1, 3, 2)            # (B, T, HW, D)

        # 2. Prepend spatial CLS -> (B, T, HW+1, D)
        tokens = torch.cat([self.spatial_cls.expand(B, -1, -1, -1), tokens], dim=2)

        # 3. Prepend temporal CLS -> (B, T+1, HW+1, D)
        tokens = torch.cat([self.temporal_cls.expand(B, -1, -1, -1), tokens], dim=1)

        # 4. Add positional embedding
        tokens = tokens + self.pos_embed

        # 5. M transformer blocks
        attn_maps = []
        for blk in self.blocks:
            if return_attn:
                tokens, t_attn, s_attn = blk(tokens, return_attn=True)
                attn_maps.append((t_attn, s_attn))
            else:
                tokens = blk(tokens)

        tokens = self.norm(tokens)

        # 6. Predict from O(0,0,:) — joint CLS token (paper Sec. III-A)
        cls_out = tokens[:, 0, 0, :]
        logits  = self.head(cls_out)

        if return_attn:
            return logits, attn_maps
        return logits

print("ISTVT defined ✓")

ISTVT defined ✓


## Sanity Check — Shape Test

In [ ]:
model = ISTVT(img_size=128, seq_len=6, embed_dim=256, depth=6, num_heads=8)
x     = torch.randn(2, 6, 3, 128, 128)
out   = model(x)

print(f"Input  shape : {x.shape}")
print(f"Output shape : {out.shape}")        # expect (2, 1)
print(f"Total params : {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Input  shape : torch.Size([2, 6, 3, 128, 128])
Output shape : torch.Size([2, 1])
Total params : 6.87M
